In [1]:
%cd /glade/u/home/zeyuanhu/ClimCorrector/models/swintransformer_v2

/glade/u/home/zeyuanhu/ClimCorrector/models/swintransformer_v2


/glade/work/zeyuanhu/mamba/climcorr/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
import torch
import numpy as np
import modulus
from utils.data_utils import *
import torch.nn as nn
from tqdm import tqdm
from swintransformer_modulus import SwinTransformerV2CrModulus
import swintransformer_modulus as swintransformer_modulus

Below I defined a wrapper function. Note that our trained NN model takes normalized input and target, but in CESM, I want the NN model to be able to take raw input and generate raw prediction without normalization. So this wrapper function basically do the job of preprocessing the input, run NN inference, and then un-normalize the NN prediction into raw correction tendencies. Note that I directly add the prune top 1st level in the postprocessing step in the wrapper.

In [5]:
class NewModel(nn.Module):
    def __init__(self, original_model, input_mean, input_std, target_mean, target_std):
        super(NewModel, self).__init__()
        self.original_model = original_model
        self.input_mean = torch.tensor(input_mean, dtype=torch.float32).view(-1, 1, 1)
        input_std[26*4:26*6] = np.maximum(input_std[26*4:26*6], 1e-5)
        self.input_std = torch.tensor(input_std, dtype=torch.float32).view(-1, 1, 1)
        self.target_mean = torch.tensor(target_mean, dtype=torch.float32).view(-1, 1, 1)
        self.target_std = torch.tensor(target_std, dtype=torch.float32).view(-1, 1, 1)
        self.v3_index = list(range(0, 104)) + list(range(26 * 6, 26 * 7)) + [182, 187, 190, 191, 192] + list(range(193, 200))


    def preprocessing(self, x):
        lat = x[:, -4]
        lon = x[:, -3]
        tod = x[:, -2]
        toy = x[:, -1]

        # Normalize the input
        x = (x - self.input_mean) / self.input_std

        # Latitude normalization
        lat_norm = lat / 90.0

        # Longitude encoding (cosine and sine)
        lon_cos = torch.cos(lon / 360.0 * 2 * torch.pi)
        lon_sin = torch.sin(lon / 360.0 * 2 * torch.pi)

        # Time of day (tod) encoding (cosine and sine)
        tod_cos = torch.cos(tod / 24.0 * 2 * torch.pi)
        tod_sin = torch.sin(tod / 24.0 * 2 * torch.pi)

        # Time of year (toy) encoding (cosine and sine)
        toy_cos = torch.cos(toy / 365.0 * 2 * torch.pi)
        toy_sin = torch.sin(toy / 365.0 * 2 * torch.pi)

        # Concatenate normalized and encoded features
        encoded_features = torch.stack(
            (lat_norm, lon_cos, lon_sin, tod_cos, tod_sin, toy_cos, toy_sin), dim=1
        )

        x = torch.cat((x[:, :-4], encoded_features), dim=1)

        x = x[:,self.v3_index]
        return x

    def postprocessing(self, x):
        # Note that here I manually prune the top 1st layer's correction tendency to 0
        x = x*self.target_std + self.target_mean
        x[:,0,:,:] = 0.0
        x[:,26,:,:] = 0.0
        x[:,52,:,:] = 0.0
        x[:,78,:,:] = 0.0
        return x

    def forward(self, x):
        x = self.preprocessing(x)
        x = self.original_model(x)
        x = self.postprocessing(x)    
        return x

In [6]:
input_mean = np.load('/glade/u/home/zeyuanhu/ClimCorrector/preprocessing/normalization/inputs/input_mean_v2_iter2_40year_sub23.npy')
input_std = np.load('/glade/u/home/zeyuanhu/ClimCorrector/preprocessing/normalization/inputs/input_std_v2_iter2_40year_sub23.npy')
target_mean_dc = np.load('/glade/u/home/zeyuanhu/ClimCorrector/preprocessing/normalization/outputs/target_dc_mean_v2_iter2_40year_sub23.npy')
target_std_dc = np.load('/glade/u/home/zeyuanhu/ClimCorrector/preprocessing/normalization/outputs/target_dc_std_v2_iter2_40year_sub23.npy')
target_mean_sum = np.load('/glade/u/home/zeyuanhu/ClimCorrector/preprocessing/normalization/outputs/target_sum_mean_v2_iter2_40year_sub23.npy')
target_std_sum = np.load('/glade/u/home/zeyuanhu/ClimCorrector/preprocessing/normalization/outputs/target_sum_std_v2_iter2_40year_sub23.npy')

In [7]:
# below the new_model will be the wrapper model

device = torch.device("cpu")
f_torch_model = "/glade/campaign/univ/uhar0026/zeyuanhu/tutorial/saved_models/swinv3_halfforce2/ckpt/ckpt_epoch_1_metric_0.1775.mdlus"
model_inf = modulus.Module.from_checkpoint(f_torch_model).to(device)
new_model = NewModel(model_inf, input_mean, input_std, target_mean_sum, target_std_sum)

/glade/work/zeyuanhu/mamba/climcorr/lib/python3.10/site-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3637.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Next we need to convert a torch model into torchscript, which will be able to use in coupled fortran-pytorch code (like our hybrid CESM).

In [8]:
NewModel.device = "cpu"
device = torch.device("cpu")
scripted_model = torch.jit.script(new_model)
scripted_model = scripted_model.eval()
saved_folder = '/glade/campaign/univ/uhar0026/zeyuanhu/tutorial/saved_models_wrapper'
save_file_torch = os.path.join(f'{saved_folder}/swinv3_halfforce2_ckpt_epoch_1_rmtop1l.pt')
scripted_model.save(save_file_torch)